In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_parquet("data/eda.parquet")
df = df.sort_index()

In [6]:
df["lag_1"] = df["Appliances"].shift(1)
df["lag_6"] = df["Appliances"].shift(6)
df["lag_144"] = df["Appliances"].shift(144)

In [7]:
df["rolling_mean_6"] = df["Appliances"].shift(1).rolling(6).mean()
df["rolling_std_6"] = df["Appliances"].shift(1).rolling(6).std()

df["rolling_mean_144"] = df["Appliances"].shift(1).rolling(144).mean()
df["rolling_std_144"] = df["Appliances"].shift(1).rolling(144).std()

In [9]:
df[["Appliances", "rolling_mean_6", "rolling_std_6"]].head(10)

,Appliances,rolling_mean_6,rolling_std_6
date,,,
2016-01-11 17:00:00,60,NaN,NaN
2016-01-11 17:10:00,60,NaN,NaN
2016-01-11 17:20:00,50,NaN,NaN
2016-01-11 17:30:00,50,NaN,NaN
2016-01-11 17:40:00,60,NaN,NaN
2016-01-11 17:50:00,50,NaN,NaN
2016-01-11 18:00:00,60,55.000000,5.477226
2016-01-11 18:10:00,60,55.000000,5.477226
2016-01-11 18:20:00,60,55.000000,5.477226


In [11]:
df["hour"] = df.index.hour
df["day_of_week"] = df.index.dayofweek
df["is_weekend"] = (df.index.dayofweek >= 5).astype(int)

In [13]:
df["diff_1"] = df["Appliances"].shift(1).diff(1)
df["diff_6"] = df["Appliances"].shift(1).diff(6)
df["diff_144"] = df["Appliances"].shift(1).diff(144)

In [15]:
df.isna().sum()[df.isna().sum() > 0]

lag_1                 1
lag_6                 6
lag_144             144
rolling_mean_6        6
rolling_std_6         6
rolling_mean_144    144
rolling_std_144     144
diff_1                2
diff_6                7
diff_144            145
dtype: int64

In [16]:
df = df.dropna()

In [17]:
print(df.isna().sum().sum())
print(df.duplicated().sum())

0
0


In [ ]:
from sklearn.model_selection import train_test_split

train, val_and_test = train_test_split(df, shuffle=False, test_size=0.3)
validation, test    = train_test_split(val_and_test, shuffle=False, test_size=0.5)
print(f"{len(train) = }\n{len(validation) = }\n{len(test) = }")

df = df.sort_index()

In [ ]:
X_train = train_df.drop(columns=["Appliances"])
X_val = val_df.drop(columns=["Appliances"])
X_test = test_df.drop(columns=["Appliances"])

y_train = train_df["Appliances"]
y_val = val_df["Appliances"]
y_test = test_df["Appliances"]

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
import numpy as np

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(y_train.to_numpy().reshape(-1, 1)).ravel()
y_val_scaled = y_scaler.transform(y_val.to_numpy().reshape(-1, 1)).ravel()
y_test_scaled = y_scaler.transform(y_test.to_numpy().reshape(-1, 1)).ravel()

In [ ]:
def create_sequences(X, y, window=144):
    history = np.column_stack((X, y))
    X_seq = np.array([history[i-window:i] for i in range(window, len(y))])
    y_seq = y[window:]
    return X_seq, y_seq